# Polymarket Research: Markets + Trades Walkthrough

It shows how to:
1. Resolve markets from a Polymarket link.
2. Extract **meta information** for broad market selection.
3. Download **full trade history** by `condition_id` or directly from URL.
4. Understand output schemas and save results.


## What You Will Get

By the end of the notebook, you will have:
- A market-universe summary (activity, volume, liquidity, spread).
- A ranked shortlist of potentially attractive markets.
- Full historical trades for a selected market.
- Exportable parquet files for downstream analysis.


In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "clients").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT


PosixPath('/Users/anuar.aimoldin/projects/polymarket_research')

In [2]:
from typing import Any

import pandas as pd

from clients.gamma_client import GammaClient
from collectors.markets_collector import MarketsCollector
from collectors.trades_collector import TradesCollector

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


## 1) Resolve Markets From a Polymarket Link

Use an `/event/...` or `/market/...` link.
- `/event/...` can map to multiple markets.
- `/market/...` usually maps to one market.


In [3]:
gamma = GammaClient()

url = "https://polymarket.com/event/fed-decision-in-march-885"
markets = gamma.resolve_markets_from_polymarket_url(url)

markets_preview = pd.DataFrame(
    {
        "market_index": list(range(len(markets))),
        "slug": [m.get("slug") for m in markets],
        "condition_id": [m.get("conditionId") or m.get("condition_id") for m in markets],
        "question": [m.get("question") for m in markets],
    }
)
markets_preview


,market_index,slug,condition_id,question
0,0,will-the-fed-decrease-interest-rates-by-50-bps...,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,Will the Fed decrease interest rates by 50+ bp...
1,1,will-the-fed-decrease-interest-rates-by-25-bps...,0xd6c49a94948e365aa74c09eef54d907b7ca534b243c3...,Will the Fed decrease interest rates by 25 bps...
2,2,will-there-be-no-change-in-fed-interest-rates-...,0x257b18205f908aef01ef2d1d50e6fea7d29cf5486fe0...,Will there be no change in Fed interest rates ...
3,3,will-the-fed-increase-interest-rates-by-25-bps...,0x25aa90b3cd98305e849189b4e8b770fc77fe89bccb7c...,Will the Fed increase interest rates by 25+ bp...


## 2) Extract Meta Information for Market Selection

`MarketsCollector.download_market_meta(...)` returns three objects:
- `markets`: full downloaded market universe frame.
- `summary`: aggregated statistics for the downloaded universe.
- `top_markets`: ranked shortlist (score blends liquidity/volume/spread).

Important:
- `top_n` limits only the **ranked output size** (`top_markets`).
- It does **not** limit how many markets are downloaded.
- Use `include_inactive=False`, filters, or `max_pages` to keep collection fast.


In [4]:
mc = MarketsCollector(gamma)

# Not finished questions
open_q = mc.download_market_meta(
    closed="false",
    include_inactive=False,
    top_n=30,
    show_progress=True,
    estimate_total=True,
)
open_q

{'markets':             id                                           question                                       condition_id                                      slug  \
 0       517310               Will Trump deport less than 250,000?  0xaf9d0e448129a9f657f851d49495ba4742055d80e0ef...        will-trump-deport-less-than-250000   
 1       517311          Will Trump deport 250,000-500,000 people?  0x49686d26fb712515cd5e12c23f0a1c7e10214c7faa3c...    will-trump-deport-250000-500000-people   
 2       517313         Will Trump deport 500,000-750,000- people?  0x2393ed0b0fdc450054c7b9071907eca75cf4fc36e385...    will-trump-deport-500000-750000-people   
 3       517314        Will Trump deport 750,000-1,000,000 people?  0x44f08744458b8896620cd3330bc5e1ea69df4199b02d...   will-trump-deport-750000-1000000-people   
 4       517315      Will Trump deport 1,000,000-1,250,000 people?  0x49a20c7523c271099008f3ef9a31521263b24d637959...  will-trump-deport-1000000-1250000-people   
 ...       

In [5]:
open_q.keys()

dict_keys(['markets', 'summary', 'top_markets'])

In [6]:
open_q['top_markets'].head()

,condition_id,slug,question,active,closed,liquidity,volume_24h,volume_1w,volume_total,spread,market_score,end_date
0,0x0b4cc3b739e1dfe5d73274740e7308b6fb389c5af040...,will-jesus-christ-return-before-2027,Will Jesus Christ return before 2027?,True,False,1.008621e+07,9.237599e+06,2.563099e+07,2.896052e+07,0.001,0.998017,2026-12-31 00:00:00+00:00
1,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,will-the-fed-decrease-interest-rates-by-50-bps...,Will the Fed decrease interest rates by 50+ bp...,True,False,1.726580e+06,2.313017e+06,1.932139e+07,5.746935e+07,0.001,0.997692,2026-03-18 00:00:00+00:00
2,0x5e5c9dfaf695371a0cc321b47b35f66a6dbd1482f050...,will-bitcoin-reach-150k-in-february-2026,"Will Bitcoin reach $150,000 in February?",True,False,1.873984e+06,2.522182e+06,1.099703e+07,2.013571e+07,0.001,0.997572,2026-03-01 05:00:00+00:00
3,0x46d40e851b24d9b0af4bc1942ccd86439cae82a90117...,will-trump-nominate-judy-shelton-as-the-next-f...,Will Trump nominate Judy Shelton as the next F...,True,False,2.312756e+06,1.349954e+06,1.451015e+07,8.848474e+07,0.001,0.997569,2026-12-31 00:00:00+00:00
4,0x25aa90b3cd98305e849189b4e8b770fc77fe89bccb7c...,will-the-fed-increase-interest-rates-by-25-bps...,Will the Fed increase interest rates by 25+ bp...,True,False,1.038870e+06,2.135126e+06,1.586742e+07,4.857207e+07,0.001,0.997225,2026-03-18 00:00:00+00:00


In [13]:
short_view_columns = ['question', 'outcomes', 'outcome_prices']
open_q['markets'][short_view_columns].head()

,question,outcomes,outcome_prices
0,"Will Trump deport less than 250,000?","[""Yes"", ""No""]","[""0.0425"", ""0.9575""]"
1,"Will Trump deport 250,000-500,000 people?","[""Yes"", ""No""]","[""0.8745"", ""0.1255""]"
2,"Will Trump deport 500,000-750,000- people?","[""Yes"", ""No""]","[""0.039"", ""0.961""]"
3,"Will Trump deport 750,000-1,000,000 people?","[""Yes"", ""No""]","[""0.014"", ""0.986""]"
4,"Will Trump deport 1,000,000-1,250,000 people?","[""Yes"", ""No""]","[""0.004"", ""0.996""]"


### Market Meta Columns (Top Markets)

| Column | Meaning |
|---|---|
| `condition_id` | Polymarket market identifier used for historical trade download. |
| `slug` | Human-readable market slug. |
| `question` | Market question text. |
| `active` | Whether market is currently active. |
| `closed` | Whether market is closed. |
| `liquidity` | Liquidity measure from Gamma payload. |
| `volume_24h` | 24-hour traded volume. |
| `volume_1w` | 1-week traded volume. |
| `volume_total` | Total volume. |
| `spread` | Market spread (lower is generally tighter). |
| `market_score` | Internal ranking score used for shortlist ordering. |
| `end_date` | Market end/resolution date (if available). |


## 3) Download Full Trades by `condition_id`

Use this when you already selected a market from `markets_preview` / `top_markets_df`.
This path uses subgraph pagination to fetch **full history** (not Data API-capped history).


In [7]:
collector = TradesCollector(gamma)

market_index = 0  # change this index based on your selection
selected_market = markets[market_index]
condition_id = selected_market.get("conditionId") or selected_market.get("condition_id")
condition_id


'0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13f5915f050b615a193c20'

In [8]:
df_trades_by_id = collector.download_all_trades(
    condition_id,
    show_progress=True,
)

print("rows:", len(df_trades_by_id))
print("time range:", df_trades_by_id["timestamp_utc"].min(), "->", df_trades_by_id["timestamp_utc"].max())
df_trades_by_id.head()


Normalizing trade size by /1000000.0 (size appears to be raw base units; applying 1e6 normalization).


rows: 102536
time range: 2025-10-29 22:21:27+00:00 -> 2026-02-19 12:26:46+00:00


,timestamp_utc,price,size,outcome,transaction_hash
0,2025-10-29 22:21:27+00:00,0.16,31.25000,Yes,0xd91006abd45d7cbf97eb9a102a753bc4ea2064a1edfe...
1,2025-10-29 22:21:27+00:00,0.84,31.25000,No,0xd91006abd45d7cbf97eb9a102a753bc4ea2064a1edfe...
2,2025-10-30 05:57:03+00:00,0.04,1.04165,Yes,0xe6b19028d38176d452d2644ad9b794099d24f665c90d...
3,2025-10-30 05:57:03+00:00,0.96,1.04165,No,0xe6b19028d38176d452d2644ad9b794099d24f665c90d...
4,2025-10-30 05:57:07+00:00,0.85,1.00000,No,0x8a1c9648bdc6ee6fac6c7b9ff58290b1a3ec8a29c878...


### Trades Output Columns

| Column | Meaning |
|---|---|
| `timestamp_utc` | Trade timestamp in UTC. |
| `price` | Implied probability-like trade price. |
| `size` | Traded outcome token size. |
| `outcome` | Outcome label when resolvable (e.g., `Yes` / `No`). |
| `transaction_hash` | On-chain transaction hash (when available). |


## 4) Download Trades Directly From URL

If you have an event link, you can skip manual `condition_id` resolution:
- For multi-market events, pass `market_index` or `market_slug`.


In [ ]:
df_trades_by_url = collector.download_all_trades_from_url(
    url,
    market_index=market_index,
    show_progress=True,
)

print("rows by condition_id:", len(df_trades_by_id))
print("rows by url:", len(df_trades_by_url))
df_trades_by_url.head()


## 5) API-Level Example (`GammaClient.get_trades`)

This is useful for quick checks, but the Data API is pagination-capped.
Use `TradesCollector.download_all_trades(...)` for full-history research workflows.


In [14]:
def as_rows(payload: Any) -> list[dict[str, Any]]:
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]
    if isinstance(payload, dict):
        for k in ("data", "trades", "results", "items"):
            v = payload.get(k)
            if isinstance(v, list):
                return [x for x in v if isinstance(x, dict)]
    return []

sample_payload = gamma.get_trades(condition_id, limit=50, offset=0)
sample_rows = as_rows(sample_payload)
print("sample rows:", len(sample_rows))

pd.DataFrame(sample_rows).head(5)


sample rows: 50


,proxyWallet,side,asset,conditionId,size,price,timestamp,title,slug,icon,eventSlug,outcome,outcomeIndex,name,pseudonym,bio,profileImage,profileImageOptimized,transactionHash
0,0xef4de987efeaa1bff3ac7903df86bf4964ada4b1,SELL,4845807501995709816634047564692338949651042763...,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,13.00,0.992,1771504824,Will the Fed decrease interest rates by 50+ bp...,will-the-fed-decrease-interest-rates-by-50-bps...,https://polymarket-upload.s3.us-east-2.amazona...,fed-decision-in-march-885,No,1,,,,,,0xdfab8853fe1ffd974d99d205126593a56749771826b7...
1,0xc29f16f3eb93eba8994e3173fa4413972031607e,SELL,4845807501995709816634047564692338949651042763...,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,24.77,0.992,1771504798,Will the Fed decrease interest rates by 50+ bp...,will-the-fed-decrease-interest-rates-by-50-bps...,https://polymarket-upload.s3.us-east-2.amazona...,fed-decision-in-march-885,No,1,Gubse,Acclaimed-Antling,,,,0x04579f2cf766575d86ad3f036a74ebcbdab9dbe4ca2b...
2,0xf754d178ab63cb67915d61b2129fab5192b0444f,SELL,4845807501995709816634047564692338949651042763...,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,17.00,0.992,1771504790,Will the Fed decrease interest rates by 50+ bp...,will-the-fed-decrease-interest-rates-by-50-bps...,https://polymarket-upload.s3.us-east-2.amazona...,fed-decision-in-march-885,No,1,,,,,,0x2e0f5f746b2cae008d7550c5c21c2277ab0a06f40e92...
3,0xaf692de1b5271d331b58d319dd7de9b1260745dc,SELL,4845807501995709816634047564692338949651042763...,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,35.00,0.992,1771504782,Will the Fed decrease interest rates by 50+ bp...,will-the-fed-decrease-interest-rates-by-50-bps...,https://polymarket-upload.s3.us-east-2.amazona...,fed-decision-in-march-885,No,1,,,,,,0x94c0d4fef691e58f0b4958009f21f66fb337dfb29f30...
4,0xaf692de1b5271d331b58d319dd7de9b1260745dc,SELL,4655345557056451798919102345870537152143651426...,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,35.00,0.007,1771504778,Will the Fed decrease interest rates by 50+ bp...,will-the-fed-decrease-interest-rates-by-50-bps...,https://polymarket-upload.s3.us-east-2.amazona...,fed-decision-in-march-885,Yes,0,,,,,,0x48baea854d3d3f6d8a3de77fe7a9b50ae8ab29d6e2ee...


## 6) Save Outputs

Persist both market meta and trades for reuse in analysis pipelines.


In [13]:
import json

cache_dir = REPO_ROOT / "data" / "demo_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

trades_cache_path = cache_dir / "selected_trades.parquet"
context_cache_path = cache_dir / "selected_market_context.json"

collector.save_to_parquet(df_trades_by_id, trades_cache_path)

context = {
    "condition_id": condition_id if "condition_id" in locals() else None,
    "question": question if "question" in locals() else None,
    "slug": slug if "slug" in locals() else None,
}
context_cache_path.write_text(json.dumps(context, ensure_ascii=False, indent=2), encoding="utf-8")

{
    "trades": str(trades_cache_path),
    "context": str(context_cache_path),
    "rows": int(len(df_trades_by_id)),
    "condition_id": context.get("condition_id"),
}


{'trades': '/Users/anuar.aimoldin/projects/polymarket_research/data/demo_cache/selected_trades.parquet',
 'context': '/Users/anuar.aimoldin/projects/polymarket_research/data/demo_cache/selected_market_context.json',
 'rows': 102536,
 'condition_id': '0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13f5915f050b615a193c20'}

## Next Steps

- Tune ranking thresholds (`min_liquidity`, `min_volume_24h`) for your strategy.
- Add `start_date` in trade download for rolling-window datasets.
- Join `top_markets_df` with your alpha/risk features for market selection.
